# 01 · Contextual Retrieval job (one-off, ~80–90 min on a T4)

Runs **A4 CONTEXTUALIZE** from `RAG_CONCEPTS.md` / `docs/02-architecture.md`: Qwen2.5-3B-Instruct on vLLM writes one
context line per chunk, **one prompt per document** (document sent once, with inline `[[c1]]…[[cN]]` markers).

- Input: `artifacts/chunks.parquet` (committed in the repo, so chunk ids are identical everywhere).
- Output: `contexts.jsonl` `{chunk_id, context}`. It's written to Google Drive if mounted, so the job is **resumable** after a
  Colab disconnect: just re-run all cells and finished chunks are skipped.
- vLLM runs in its **own venv as a background subprocess**: it pins its own torch, and its GPU memory is released on exit.

When it's done, download `contexts.jsonl` and commit it to `artifacts/`. The demo notebook (`02_demo_and_eval.ipynb`) never re-runs this.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import os, subprocess, json, time
REPO = "https://github.com/nrusnac/DP_task_RAG.git"
if not os.path.exists("/content/DP_task_RAG"):
    subprocess.run(["git", "clone", "-q", REPO, "/content/DP_task_RAG"], check=True)
os.chdir("/content/DP_task_RAG")
subprocess.run(["git", "pull", "-q"], check=True)
!pip -q install -e .

In [ ]:
# Optional but recommended: write the output to Drive so a disconnect doesn't lose progress
OUT_DIR = "/content/DP_task_RAG/artifacts"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/dp_task_rag"
    os.makedirs(OUT_DIR, exist_ok=True)
except Exception as e:
    print("Drive not mounted, writing locally:", e)
OUT = f"{OUT_DIR}/contexts.jsonl"
print("output ->", OUT)

In [ ]:
!python -m hrag.ingest.contextualize prepare

In [ ]:
# vLLM in an isolated venv (~2 min)
if not os.path.exists("/content/vllm_env/bin/python"):
    !pip -q install uv
    !uv venv -q /content/vllm_env --python 3.12
    !uv pip install -q --python /content/vllm_env/bin/python vllm
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# Launch in the background: this cell returns immediately and the job keeps running while the runtime is alive
log = open("/content/ctx.log", "a")
proc = subprocess.Popen(["/content/vllm_env/bin/python", "scripts/contextualize_vllm.py", "data/index/ctx_input.json", OUT],
                        stdout=log, stderr=subprocess.STDOUT, cwd="/content/DP_task_RAG")
print("started pid", proc.pid, "| log: /content/ctx.log")

Re-run the cell below any time to check progress (the first ~3 min are model loading and compilation).

In [ ]:
status = "running" if proc.poll() is None else f"exited with code {proc.returncode}"
print("status:", status)
!grep -E "jobs|Error|error" /content/ctx.log | tail -4
n = sum(1 for _ in open(OUT)) if os.path.exists(OUT) else 0
print(f"chunks with a context row: {n} / 5756")
if proc.poll() not in (None, 0):
    !tail -40 /content/ctx.log

In [ ]:
# When finished: download the file and commit it to artifacts/contexts.jsonl
from google.colab import files
files.download(OUT)